In [0]:
# =============================================================================
# Notebook 01 — Build Master Dataset
# Goal     : Rename columns (_c0…_c77) → original names and filter
#             use_h2_main == True → ml_perovskites.master (691 rows)
# Input    : ml_perovskites.db_ml_curated  (uploaded via UI without headers)
# Output   : Delta table ml_perovskites.master
# Seed     : N/A (no random operations in this step)
# =============================================================================

# ── 0. Imports ────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
import pandas as pd

# ── 1. Load raw table ─────────────────────────────────────────────────────────
raw = spark.table("ml_perovskites.db_ml_curated")

print(f"Total rows (incl. header row): {raw.count()}")
print(f"Columns                      : {len(raw.columns)}")

# ── 2. Column mapping: _cN → original name ───────────────────────────────────
# Databricks UI upload did not preserve column headers.
# Row 0 of each column contained the actual column name.
# Mapping derived from notebook 01a_inspect_schema.py output.

col_map = {
    "_c0":  "source_row_id",
    "_c1":  "Compound_original",
    "_c2":  "base_compound",
    "_c3":  "family_proposed",
    "_c4":  "doi",
    "_c5":  "doi_key",
    "_c6":  "h2_rate_umol_g_h",
    "_c7":  "h2_umol_h_rate_mass",
    "_c8":  "bandgap_eV",
    "_c9":  "particle_size_nm",
    "_c10": "ssa_m2_g",
    "_c11": "lamp_power_W",
    "_c12": "cutoff_nm",
    "_c13": "catalyst_mass_g",
    "_c14": "cocatalyst",
    "_c15": "cocatalyst_class",
    "_c16": "cocatalyst_precursor",
    "_c17": "reaction_type_original",
    "_c18": "solution_components",
    "_c19": "scavenger_class",
    "_c20": "organic_vol_fraction",
    "_c21": "prep_class",
    "_c22": "prep_multistep",
    "_c23": "prep_method_original",
    "_c24": "thermal_T_C",
    "_c25": "thermal_T_le_250C",
    "_c26": "thermal_time_h",
    "_c27": "y_log10_1p",
    "_c28": "rate_db_original",
    "_c29": "rate_final_umol_g_h",
    "_c30": "rate_status",
    "_c31": "rate_rule",
    "_c32": "rate_confidence",
    "_c33": "rate_candidate_umol_g_h",
    "_c34": "use_h2",
    "_c35": "rate_note",
    "_c36": "flag_cutoff",
    "_c37": "flag_bandgap",
    "_c38": "flag_ssa",
    "_c39": "flag_size",
    "_c40": "flag_thermal",
    "_c41": "flag_mass",
    "_c42": "flag_cocatalyst",
    "_c43": "flag_solution",
    "_c44": "flag_compound",
    "_c45": "field_discrepancies_text",
    "_c46": "A_site_paper",
    "_c47": "B_site_paper",
    "_c48": "dopant_paper",
    "_c49": "dopant_fraction_paper",
    "_c50": "anion_paper",
    "_c51": "secondary_phase_paper",
    "_c52": "formula_base_paper",
    "_c53": "run_type",
    "_c54": "substrate_is_target",
    "_c55": "scavenger_class_v1",
    "_c56": "domain",
    "_c57": "structure_review",
    "_c58": "scope_main",
    "_c59": "use_h2_main",
    "_c60": "censored_trace",
    "_c61": "y_log10_1p_final",
    "_c62": "source_type",
    "_c63": "cutoff_nm_db",
    "_c64": "cutoff_status",
    "_c65": "cutoff_nm_final",
    "_c66": "cutoff_evidence",
    "_c67": "cocatalyst_loading_paper",
    "_c68": "volume_mL",
    "_c69": "irradiance_mW_cm2",
    "_c70": "pH",
    "_c71": "T_reaction_C",
    "_c72": "reaction_time_h",
    "_c73": "AQY_paper",
    "_c74": "scavenger_conc",
    "_c75": "calcination_note",
    "_c76": "sample_in_paper",
    "_c77": "AQY_note",
}

# Rename columns
renamed = raw.toDF(*[col_map.get(c, c) for c in raw.columns])

# ── 3. Remove header row (source_row_id == 'source_row_id') ──────────────────
renamed = renamed.filter(F.col("source_row_id") != "source_row_id")
print(f"\nRows after removing header row : {renamed.count()}")

# ── 4. Main filter: use_h2_main == True ──────────────────────────────────────
# Column is string ("true"/"false") because upload had no type inference.
master = renamed.filter(F.lower(F.col("use_h2_main").cast("string")) == "true")
print(f"Rows after use_h2_main filter  : {master.count()}")

# ── 5. Cast essential numeric columns ────────────────────────────────────────
numeric_cols = [
    "rate_final_umol_g_h", "y_log10_1p_final", "y_log10_1p",
    "bandgap_eV", "particle_size_nm", "ssa_m2_g",
    "lamp_power_W", "cutoff_nm", "catalyst_mass_g",
    "thermal_T_C", "thermal_time_h", "organic_vol_fraction",
    "volume_mL", "irradiance_mW_cm2", "pH", "T_reaction_C",
]

for col in numeric_cols:
    if col in master.columns:
        # try_cast returns NULL for malformed strings instead of raising an error
        master = master.withColumn(col, F.expr(f"try_cast(`{col}` as double)"))

# ── 6. Null checks on key columns ────────────────────────────────────────────
required_non_null = ["rate_final_umol_g_h", "doi", "source_row_id"]
print("\nNull checks on key columns:")
for col in required_non_null:
    n_null = master.filter(F.col(col).isNull()).count()
    print(f"  NaN in '{col}': {n_null}")

# ── 7. Diagnostic distributions ──────────────────────────────────────────────
print("\nDistribution by domain:")
master.groupBy("domain").count().orderBy("count", ascending=False).show(20, truncate=False)

print("\nDistribution by family_proposed:")
master.groupBy("family_proposed").count().orderBy("count", ascending=False).show(20, truncate=False)

print("\nDistribution by run_type:")
master.groupBy("run_type").count().orderBy("count", ascending=False).show(20, truncate=False)

# ── 8. Rate distribution histogram ───────────────────────────────────────────
master_pd = master.select(
    "source_row_id", "doi", "rate_final_umol_g_h",
    "y_log10_1p_final", "run_type", "scavenger_class",
    "cocatalyst_class", "domain", "family_proposed", "rate_confidence"
).toPandas()

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(master_pd["rate_final_umol_g_h"].dropna(), bins=60, color="#4C72B0", edgecolor="white")
axes[0].set_xlabel("rate_final_umol_g_h (µmol/g/h)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution — linear scale")

log_vals = np.log10(1 + master_pd["rate_final_umol_g_h"].dropna())
axes[1].hist(log_vals, bins=60, color="#55A868", edgecolor="white")
axes[1].set_xlabel("log₁₀(1 + rate)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution — log scale")

plt.tight_layout()

# Save to /tmp (writable on all Databricks editions, including Community)
save_path = "/tmp/01_rate_distribution.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to {save_path}")

# ── 9. Write Delta table ──────────────────────────────────────────────────────
# Schema ml_perovskites already exists in the Databricks catalog.
(master
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.master"))

print("\n✓ Table ml_perovskites.master written successfully.")
print(f"  Rows   : {spark.table('ml_perovskites.master').count()}")
print(f"  Columns: {len(spark.table('ml_perovskites.master').columns)}")

# ── 10. Acceptance criteria ───────────────────────────────────────────────────
print("\n── Acceptance criteria ─────────────────────────────────────────────────")
final = spark.table("ml_perovskites.master")
n = final.count()
assert n > 0, "ERROR: table is empty"

for col in required_non_null:
    n_null = final.filter(F.col(col).isNull()).count()
    status = "✓" if n_null == 0 else f"✗ {n_null} nulls"
    print(f"  {col}: {status}")

print(f"\n  Total rows     : {n}")
print(f"  Distinct DOIs  : {final.select('doi').distinct().count()}")
print("────────────────────────────────────────────────────────────────────────")